# 🧪 08 - Pandas Wrangle Lab: Clean and Explore a Messy Purchases Dataset

A retail company has sent you `messy_purchases.csv`. Each row is supposed to be **one purchase**, but the file has every problem from this module:
- Prices stored as text, with dollar signs and commas (`'$1,249.99'`)
- Placeholder values like `'Unknown'`, `'not recorded'`, and `-1`
- State and department names with inconsistent capitals, spaces, and spellings (`'il '`, `'Illinois'`)
- Dates stored as text, in three different formats
- Duplicate purchases, and messy column names
- A column that isn't needed for the analysis

Your job: produce a **clean** dataset you can trust, then answer business questions about **revenue by state and department**. Work through the sections in order; each one uses a notebook from this module.

## ✅ Learning Goals
- Practice loading, cleaning, and exploring real-world-ish data, from start to finish
- Detect and handle missing values, placeholders, duplicates, and type mismatches
- Apply column creation, renaming, conditional updates, sorting, and filtering together
- Use `.info()`, `.describe()`, `.value_counts()`, and `.query()` fluently
- Group and summarize with `.groupby()`, then validate and export the results

```text
Load → Inspect → Standardize text and placeholders → Fix types
     → Handle missing values and duplicates → Create columns
     → Filter, sort, and summarize → Validate and export
```

## 0. Setup
Everything this lab saves goes into **`08-OutputFiles`** (the same pattern as Foundations). Upload that folder along with this notebook.

In [ ]:
import pandas as pd
from pathlib import Path

OUT = Path('08-OutputFiles')
OUT.mkdir(exist_ok=True)

## 1. Load and Look (Change Nothing Yet)
Inspect before you clean (notebook 02). Don't change anything in this section, just look.

In [ ]:
purchases = pd.read_csv('messy_purchases.csv')
purchases.head()

In [ ]:
print('shape:', purchases.shape)
print('columns:', purchases.columns.tolist())

In [ ]:
purchases.info()

In [ ]:
purchases.describe(include='all')

✍️ **Before going on, note your answers:**
1. How many rows and columns are there?
2. Which columns have missing values?
3. Which columns *look* numeric but are stored as text? (`Unit Price`? `QTY`?)
4. Are the dates real dates, or text?
5. Do the column names need cleaning? (Look closely at `' Department'` and `' Unit Price '`.)

*Your answers here.*

## 2. Make a Working Copy
Keep the raw data untouched, so you can compare before and after (notebook 04):

```text
raw data            → purchases
working clean copy  → clean
```

In [ ]:
clean = purchases.copy()

## 3. Standardize the Column Names
Fix the names **first**, because every later line depends on them (notebook 05):

In [ ]:
clean.columns = clean.columns.str.strip().str.lower().str.replace(' ', '_')
clean = clean.rename(columns={'qty': 'quantity'})
clean.columns.tolist()

## 4. Inspect the Categories
Text categories hide spelling differences, stray spaces, and placeholders. `value_counts(dropna=False)` shows them all:

In [ ]:
clean['state'].value_counts(dropna=False)

In [ ]:
clean['department'].value_counts(dropna=False)

Five real states appear as nearly twenty "different" values: `'IL'`, `'il'`, `'IL '`, `'Illinois'`, ...

💡 Where did `'N/A'` and the blank cells go? `pd.read_csv` **already** turns a few common placeholders, including blanks, `'N/A'`, `'NA'`, and `'null'`, into `NaN` while loading. That's why they show up as `NaN` above. It **doesn't** know about `'Unknown'`, `'not recorded'`, or `-1`. Those are up to you.

## 5. Standardize the Text
Strip spaces and fix capitalization, then map long names to codes (notebook 04). `.astype('string')` makes sure every value is text before using `.str`:

In [ ]:
for col in ['state', 'department', 'product_name']:
    clean[col] = clean[col].astype('string').str.strip()

clean['state'] = clean['state'].str.upper().replace({
    'ILLINOIS': 'IL', 'WISCONSIN': 'WI', 'MINNESOTA': 'MN', 'INDIANA': 'IN', 'IOWA': 'IA',
})
clean['department'] = clean['department'].str.title()

clean['state'].value_counts(dropna=False)

## 6. Replace Placeholders with Real Missing Values
Now that the text is standardized, the placeholders are predictable: `'UNKNOWN'` in `state` (it was upper-cased), and `'Unknown'` in `department` (title-cased). Replace them **column by column** with a nested dictionary (notebook 04):

⚠️ Standardize **before** replacing. If you replaced `'UNKNOWN'` in `department` after title-casing, nothing would match, because it's spelled `'Unknown'` by then.

In [ ]:
clean = clean.replace({
    'state':         {'UNKNOWN': pd.NA},
    'department':    {'Unknown': pd.NA},
    'unit_price':    {'-1': pd.NA},
    'quantity':      {'-1': pd.NA},
    'purchase_date': {'not recorded': pd.NA},
})

print('Missing cells in total:', clean.isna().sum().sum())
clean.isna().sum()

In [ ]:
# Rows with at least one missing value
clean[clean.isna().any(axis=1)].head()

## 7. Fix the Numbers
`unit_price` is text like `'$1,249.99'` and `' 4.99 '`. Remove the dollar signs, commas, and spaces, **then** convert. Keep the raw values, so you can see what failed (notebook 04). `regex=False` treats `$` as plain text (notebook 06):

In [ ]:
raw_price = clean['unit_price'].copy()

clean['unit_price'] = (clean['unit_price']
                       .str.replace('$', '', regex=False)
                       .str.replace(',', '', regex=False)
                       .str.strip())
clean['unit_price'] = pd.to_numeric(clean['unit_price'], errors='coerce')

print('Prices that failed to convert:', raw_price[clean['unit_price'].isna() & raw_price.notna()].tolist())

`quantity` should be whole numbers, but one row says `'two'`, just like the `'ninety'` hiding in a score column in this lab's earlier version. It's a word, not a missing value, so `.isna()` never counted it. Convert with `errors='coerce'`, and use the nullable `'Int64'` type so the column can hold missing values (notebook 04):

In [ ]:
raw_quantity = clean['quantity'].copy()
clean['quantity'] = pd.to_numeric(clean['quantity'], errors='coerce').astype('Int64')

print('Quantities that failed to convert:', raw_quantity[clean['quantity'].isna() & raw_quantity.notna()].tolist())
clean[['unit_price', 'quantity']].dtypes

## 8. Fix the Dates
The dates mix three formats (`2026-01-05`, `01/08/2026`, `Jan 8, 2026`). Watch what happens without `format='mixed'` (notebook 04):

In [ ]:
guessed = pd.to_datetime(clean['purchase_date'], errors='coerce')
mixed = pd.to_datetime(clean['purchase_date'], errors='coerce', format='mixed')

print('Missing dates, format guessed:', guessed.isna().sum())
print("Missing dates, format='mixed':", mixed.isna().sum())

Guessing the format from the first value would have thrown away most of the dates, **silently**. Use `format='mixed'`, then check the range and add a month column with `.dt` (notebook 05):

In [ ]:
clean['purchase_date'] = mixed
print('From', clean['purchase_date'].min().date(), 'to', clean['purchase_date'].max().date())

clean['purchase_month'] = clean['purchase_date'].dt.to_period('M')
clean[['purchase_date', 'purchase_month']].head(3)

## 9. Find and Remove Duplicates
Look **before** you delete (notebook 04). First, exact copies:

In [ ]:
print('Exact duplicate rows:', clean.duplicated().sum())
clean[clean.duplicated(keep=False)].sort_values('purchase_id').head(6)

In [ ]:
clean = clean.drop_duplicates()
print('Repeated purchase IDs left:', clean.duplicated(subset=['purchase_id']).sum())

clean[clean.duplicated(subset=['purchase_id'], keep=False)].sort_values('purchase_id')[
    ['purchase_id', 'product_name', 'unit_price', 'quantity', 'purchase_date']]

These rows share a `purchase_id` but have different quantities. Each **purchase** should appear once, and the company says later entries are **corrections**, so keep the last one. (Without that information, you'd need to ask! Never guess silently.)

In [ ]:
clean = clean.drop_duplicates(subset=['purchase_id'], keep='last')
print('Rows now:', len(clean))

## 10. Fill or Drop?
Decide column by column (notebook 04):

| Column | Missing means... | Decision |
|---|---|---|
| `state`, `department` | Unknown, but the purchase is still real | **Fill** with `'NOT PROVIDED'` / `'Not Provided'` |
| `unit_price`, `quantity` | Can't compute revenue | **Drop** from the revenue analysis |
| `purchase_date` | Can't place it in time | **Drop** |
| `quantity` | Not "zero items"! | Never fill with `0` |

In [ ]:
clean['state'] = clean['state'].fillna('NOT PROVIDED')
clean['department'] = clean['department'].fillna('Not Provided')

before = len(clean)
clean = clean.dropna(subset=['purchase_id', 'purchase_date', 'unit_price', 'quantity'])
print('Dropped', before - len(clean), 'rows missing an essential field. Rows now:', len(clean))

## 11. Create Useful Columns
Whole-column operations, a True/False flag, and bands with `pd.cut()` (notebook 05):

In [ ]:
clean['revenue'] = clean['unit_price'] * clean['quantity']
clean['multi_item'] = clean['quantity'] >= 2
clean['revenue_category'] = pd.cut(clean['revenue'], bins=[0, 20, 100, float('inf')],
                                   labels=['Small', 'Medium', 'Large'])

clean[['product_name', 'unit_price', 'quantity', 'revenue', 'revenue_category']].head()

## 12. Update Only Matching Rows
Set a default, then apply each rule with `.loc[]`. The **last** matching rule wins, so the more generous large-order discount goes last (notebook 05):

In [ ]:
clean['discount_rate'] = 0.00
clean.loc[clean['department'] == 'Office', 'discount_rate'] = 0.10
clean.loc[clean['revenue'] >= 100, 'discount_rate'] = 0.15

clean['discounted_revenue'] = (clean['revenue'] * (1 - clean['discount_rate'])).round(2)
clean['discount_rate'].value_counts()

💡 We filled the missing departments in section 10 **first** for a reason. In a `'string'` column, comparing a missing value gives `<NA>` instead of `False`, and `.loc[]` refuses a mask that contains `<NA>`.

## 13. Drop What You Don't Need
Build the analysis table **without** changing `clean`, so the full version is still there (notebook 05):

In [ ]:
analysis = clean.drop(columns=['supplier_note'], errors='ignore')
analysis.columns.tolist()

## 14. Explore the Clean Data
Re-run the same inspection as section 1. What changed?

In [ ]:
analysis.info()

In [ ]:
analysis[['unit_price', 'quantity', 'revenue']].describe().round(2)

In [ ]:
print(analysis['revenue_category'].value_counts().to_dict())
analysis['state'].value_counts(normalize=True).mul(100).round(1)

## 15. Filter
Masks, `.isin()`, and the rows-and-columns `.loc[]` pattern (notebooks 03 and 06):

In [ ]:
office_in_il = (analysis['department'] == 'Office') & (analysis['state'] == 'IL')
print('Office purchases in IL:', office_in_il.sum())

neighbors = analysis['state'].isin(['IL', 'WI'])
print('Purchases in IL or WI: ', neighbors.sum())

analysis.loc[analysis['revenue'] >= 500, ['purchase_id', 'product_name', 'state', 'revenue']]

## 16. Query
For long conditions, `.query()` reads like a sentence, and `@` brings in Python variables (notebook 06):

In [ ]:
target_state = 'WI'
minimum_revenue = 100

analysis.query('state == @target_state and revenue >= @minimum_revenue and quantity >= 2')[
    ['purchase_id', 'product_name', 'quantity', 'revenue']]

## 17. Sort into a Report
Filter first, then pick columns, then sort (notebook 06):

In [ ]:
top_purchases = (analysis
                 .loc[analysis['revenue'] >= 50, ['purchase_id', 'product_name', 'state', 'department', 'revenue']]
                 .sort_values('revenue', ascending=False))
top_purchases.head(10)

## 18. Group and Summarize
Now the business questions (notebook 07). First, by state:

In [ ]:
by_state = (analysis
            .groupby('state', as_index=False)
            .agg(purchases=('purchase_id', 'count'),
                 total_revenue=('revenue', 'sum'),
                 average_revenue=('revenue', 'mean'))
            .round(2)
            .sort_values('total_revenue', ascending=False))
by_state

Illinois has the most purchases, but is it the top state for **revenue**? Counting rows and adding up money can tell different stories. Now by state **and** department; each row of the result is one `(state, department)` group:

In [ ]:
by_state_department = (analysis
                       .groupby(['state', 'department'], as_index=False)
                       .agg(purchases=('purchase_id', 'count'),
                            total_revenue=('revenue', 'sum'),
                            largest_purchase=('revenue', 'max'))
                       .round(2)
                       .sort_values(['state', 'total_revenue'], ascending=[True, False]))
by_state_department.head(10)

## 19. Validate
Clean data isn't just data with fewer rows. Check that the cleaning did what you meant (notebook 04):

In [ ]:
quality = pd.DataFrame({
    'raw':   {'rows': len(purchases),
              'missing cells': purchases.isna().sum().sum(),
              'exact duplicates': purchases.duplicated().sum()},
    'clean': {'rows': len(analysis),
              'missing cells': analysis.isna().sum().sum(),
              'exact duplicates': analysis.duplicated().sum()},
})
quality

In [ ]:
print('Repeated purchase IDs:', analysis['purchase_id'].duplicated().sum())
print('Departments:', sorted(analysis['department'].unique()))
print('States:     ', sorted(analysis['state'].unique()))
analysis['revenue'].describe().round(2)

Ask yourself:
- Are the essential columns complete? Are prices numbers, and dates real dates?
- Are the placeholder values gone, and do the state and department categories look standardized?
- Were the duplicate rules appropriate? Do the revenue values look reasonable?

Notice what the raw "missing cells" count **didn't** include: `'Unknown'`, `'not recorded'`, `'-1'`, and `'two'` were hiding in plain sight until sections 6 and 7. A missing-value report on raw data only counts what pandas can recognize.

## 20. Export
Reset the row labels (rows were dropped), then save the clean data and the summary report to `OUT` (Foundations notebook 06):

In [ ]:
analysis = analysis.reset_index(drop=True)
analysis.to_csv(OUT / 'cleaned_purchases.csv', index=False)
by_state_department.to_csv(OUT / 'revenue_by_state_department.csv', index=False)
print('Saved:', sorted(p.name for p in OUT.glob('*.csv')))

## 🏋️ Practice

### Practice 1 — Verify the Cleanup (easy)
Confirm that `analysis` has **zero** missing values in `purchase_date`, `unit_price`, `quantity`, and `revenue`, and that it has fewer rows than `purchases`. Then print how many rows were removed in total, and what percentage of the raw data that is.

In [ ]:
# Your code here


### Practice 2 — Revenue Share (medium)
Using `groupby`, find each **department's** share of total revenue as a percentage (its revenue ÷ all revenue × 100), rounded to 1 decimal place and sorted from largest to smallest. Is it the same order as `analysis['department'].value_counts()`? Why or why not?

In [ ]:
# Your code here


### Practice 3 — Group by a New Bucket (harder)
Create an `order_size` column that's `'Single'` when `quantity` is 1 and `'Multiple'` otherwise. Group by it and show the number of purchases, the total revenue, and the average revenue for each group.

In [ ]:
# Your code here


## ❓ Lab Questions
Answer each in the Markdown cell below, and use the code cell for any calculations:
1. What were the original dimensions of the dataset, and the final ones?
2. Which placeholder values did `read_csv` handle automatically, and which did you replace yourself?
3. Which columns needed type conversion, and why? Which **values** failed to convert?
4. How many exact duplicate rows were there? How many repeated `purchase_id`s, and how did you decide which to keep?
5. Which state had the **most purchases**? Which generated the **most revenue**? Why might they differ?
6. Which `(state, department)` group generated the most revenue?
7. What's the average purchase revenue for each department?
8. How many purchases have revenue of at least \$100?
9. Explain why `analysis.loc[analysis['revenue'] >= 100, 'discount_rate'] = 0.15` is safer than changing a filtered copy like `analysis[analysis['revenue'] >= 100]['discount_rate'] = 0.15`.

*Your answers here.*

In [ ]:
# Your code here


## ✏️ Challenge Task
Build a report of **Travel** purchases in **Illinois** with revenue over \$50, using `.query()` with `@` variables for the department, the state, and the minimum revenue. Then:
- Only include columns: `purchase_id`, `purchase_date`, `product_name`, `quantity`, `revenue`
- Sort by `revenue`, highest first, and reset the index
- Save it as `OUT / 'travel_il_report.csv'`

In [ ]:
# Your turn here


## 🔥 Challenge (Optional) — A Second Messy Column, and More
Real datasets are rarely messy in just one place.
1. Run `analysis['product_name'].value_counts()` and look closely. One product still appears under **several spellings** that section 5 didn't fix. Standardize it, and re-check your department totals.
2. Add a `purchase_quarter` column with `.dt.quarter`, and find the total revenue per **month** and per **quarter**.
3. Find the **highest-revenue product** in each department.
4. Flag purchases above the 95th percentile of revenue as `needs_review` (*hint:* `analysis['revenue'].quantile(0.95)`).
5. Compare total revenue **before** and **after** discounts, by department.
6. Make a bar chart of total revenue by department (Foundations notebook 07), and save it to `OUT`.

In [ ]:
# Your investigation here


## 📦 Check Your Output Files
Run this last. You should see `cleaned_purchases.csv`, `revenue_by_state_department.csv`, and `travel_il_report.csv` (plus anything from the Challenge). Upload `08-OutputFiles` along with this notebook.

In [ ]:
for path in sorted(OUT.rglob('*')):
    print(path)

---
### 📝 Summary
This lab took a messy file all the way to a trustworthy report:
- **Inspected** before changing anything, and kept the raw data for comparison
- **Standardized** column names and text, **then** turned placeholders into real missing values
- **Fixed types**: prices with `$` and commas, a word in a number column, and mixed date formats (`format='mixed'`)
- **Removed duplicates** deliberately, and decided **column by column** whether to fill or drop
- **Created** revenue, flag, band, and discount columns with whole-column operations and `.loc[]`
- **Filtered, sorted, grouped, validated, and exported** the results

> Clean data isn't just data with fewer rows. It's data whose values, types, categories, and assumptions you've inspected well enough to support trustworthy analysis.